# 03.3 — The session-balanced sampler

Multi-probe ephys data comes from different recording sessions, and a session's signal has its own baseline, gain, and channel layout. Mix sessions inside one minibatch and the batch statistics become a meaningless average — batch normalization, in particular, computes garbage. This project's answer, ported from MATLAB, is a **sampler that guarantees every batch is drawn from a single session**.

This notebook covers:

* Samplers vs the `shuffle`/`batch_size` shortcut.
* `SingleSessionBatchSampler` — the algorithm and why it exists.
* `set_epoch` for reproducible-yet-varying shuffles.
* Wiring a batch sampler into a DataLoader (and the arguments it replaces).

**Prerequisite:** [03.2 DataLoader and collation](03.2_dataloader_and_collation.ipynb).

## Section 1 — What MATLAB does

The MATLAB pipeline pre-splits each session's datastore into fixed minibatches, then draws from those partitions — `cgg_procSplitSingleSessionDataStoreByMiniBatchSize` and `cgg_procAllSessionMiniBatchTable`:

```matlab
% Per session, chop that session's trials into MiniBatchSize chunks;
% a batch is always one session's chunk, never a mix.
for s = 1:NumSessions
    sessionStore = subset(DataStore, sessionIdx{s});
    miniBatches{s} = splitByMiniBatchSize(sessionStore, MiniBatchSize, WantFullBatch);
end
allBatches = shuffleTogether(miniBatches{:});   % order shuffled; composition preserved
```

Two invariants: **each batch is single-session**, and the **order** of batches is shuffled each epoch while the **composition** (which trials share a batch) can also re-roll. The Python `SingleSessionBatchSampler` reproduces both.

## Section 2 — The Python concepts you need

### 2.1 — What a sampler is

A DataLoader's batching has two layers you can override:

* a **`sampler`** yields individual indices (replacing `shuffle`),
* a **`batch_sampler`** yields whole *lists* of indices — one list per batch — owning batching entirely.

`shuffle=True` is just shorthand for "use a `RandomSampler`." When batching needs a *rule* — like "never cross a session boundary" — you write a `batch_sampler`. It's any object with `__iter__` (yields index lists) and `__len__` (batch count).

### 2.2 — The sampler in action

In [ ]:
import numpy as np
from neural_data_decoding.data.samplers import SingleSessionBatchSampler

# 3 sessions: A has 10 trials, B has 7, C has 4 — deliberately uneven.
session_ids = np.array(["A"] * 10 + ["B"] * 7 + ["C"] * 4)

sampler = SingleSessionBatchSampler(session_ids, batch_size=4, drop_last=False, seed=0)

print(f"{len(sampler)} batches from {len(session_ids)} trials\n")
for batch_indices in sampler:
    sessions_in_batch = set(session_ids[batch_indices])
    print(f"  indices {str(batch_indices):24} → session(s) {sessions_in_batch}")

Every batch lists exactly one session — that's the invariant. Note the arithmetic with `drop_last=False`: A (10) → batches of 4+4+2, B (7) → 4+3, C (4) → 4. Uneven sessions produce ragged final batches per session, never a cross-session batch to "fill up."

Flip `drop_last=True` and the ragged tails vanish (MATLAB's `WantFullBatch=true`):

In [ ]:
strict = SingleSessionBatchSampler(session_ids, batch_size=4, drop_last=True, seed=0)
for batch_indices in strict:
    print(f"  {str(batch_indices):18} → {set(session_ids[batch_indices])} (size {len(batch_indices)})")
print(f"\n{len(strict)} full batches — the 2-trial A tail and 3-trial B tail were dropped")

### 2.3 — `set_epoch`: same seed, different-each-epoch order

Reproducibility ([00.8 §5](../00_orientation/00.8_build_a_dl_project_from_scratch.ipynb)) wants deterministic runs; good training wants a *different* shuffle each epoch. The reconciliation is `set_epoch(e)` — the shuffle is a deterministic function of `(seed, epoch)`, so epoch 0 always looks one way, epoch 1 another, and the whole run replays exactly given the seed:

In [ ]:
s = SingleSessionBatchSampler(session_ids, batch_size=4, seed=0)

def first_batch(sampler):
    return list(next(iter(sampler)))

s.set_epoch(0); e0 = first_batch(s)
s.set_epoch(1); e1 = first_batch(s)
s.set_epoch(0); e0_again = first_batch(s)

print("epoch 0 first batch:", e0)
print("epoch 1 first batch:", e1, " ← different order")
print("epoch 0 again:      ", e0_again, " ← identical to first time (reproducible)")

The training loop calls `sampler.set_epoch(epoch)` at the top of each epoch — one line that buys both properties.

### 2.4 — Wiring it into a DataLoader

A batch sampler is *mutually exclusive* with `batch_size`/`shuffle`/`drop_last` (the error from [03.2 §5](03.2_dataloader_and_collation.ipynb)) — it owns all of that:

In [ ]:
import torch
from torch.utils.data import DataLoader
from neural_data_decoding.data import SyntheticTrialDataset
from neural_data_decoding.data.dataset import collate_trials

ds = SyntheticTrialDataset(
    num_sessions=3, trials_per_session=8, num_samples=6,
    num_features=4, num_classes_per_dim=[3], seed=0,
)
sampler = SingleSessionBatchSampler(ds.session_ids, batch_size=4, seed=0)

loader = DataLoader(ds, batch_sampler=sampler, collate_fn=collate_trials)
#                       ^^^^^^^^^^^^^^^^^^^^^  NOT batch_size/shuffle/drop_last

for batch in loader:
    sess = {m["session_id"] for m in batch["metadata"]}
    print(f"batch x{tuple(batch['x'].shape)} from session_id(s) {sess}")

`ds.session_ids` — a property every dataset in this project exposes ([01.4 §2.5](../01_python_for_matlab_users/01.4_classes_and_oop.ipynb) was the `@property` that makes it look like an attribute) — is what the sampler groups on. For `MatFileTrialDataset` it comes from each trial's `Target.SessionName`; for `SyntheticTrialDataset` from the generation config. Same interface either way.

## Section 3 — The `neural_data_decoding` implementation

The grouping + batching core:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/data/samplers.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if line.strip().startswith("def __iter__"))
for k in range(i, min(i + 30, len(src))):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* One RNG seeded by `(self._seed, self._epoch)` drives both shuffles — the reproducible-yet-varying property of §2.3, in code.
* Per-session index lists are chunked (`batch_size` at a time), the chunks pooled across sessions, then the *chunk order* shuffled — the MATLAB `shuffleTogether` step. A chunk never spans sessions because chunking happens *within* each session's list.
* `drop_last` decides the fate of each session's trailing partial chunk — checked per session, so a small session isn't unfairly dropped whole.
* The docstring cites the MATLAB source (`cgg_procSplitSingleSessionDataStoreByMiniBatchSize`) and the `WantFullBatch` flag it maps to.

## Section 4 — Hands-on exercises

### Exercise 4.1 — prove the invariant

Build a sampler over 5 sessions of random sizes and assert — over every batch — that no batch mixes sessions. (This is essentially the property test in `tests/unit/test_samplers.py`.)

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
rng = np.random.default_rng(1)
sizes = rng.integers(3, 12, size=5)
sids = np.concatenate([[f"S{i}"] * n for i, n in enumerate(sizes)])
smp = SingleSessionBatchSampler(sids, batch_size=4, seed=7)
assert all(len(set(sids[b])) == 1 for b in smp), "a batch mixed sessions!"
print(f"{len(sizes)} sessions, sizes {sizes.tolist()}, {len(smp)} batches — invariant holds ✓")

### Exercise 4.2 — count the batches

With sessions of size `[10, 7, 4]` and `batch_size=4`: derive the batch count for `drop_last=False` and `True` by hand (per-session ceil vs floor), then check against `len(sampler)`.

In [ ]:
# Reveal:
sids = np.array(["A"]*10 + ["B"]*7 + ["C"]*4)
import math
loose = sum(math.ceil(n/4) for n in (10, 7, 4))    # 3+2+1 = 6
tight = sum(n // 4 for n in (10, 7, 4))              # 2+1+1 = 4
print("predicted:", loose, tight)
print("actual:   ",
      len(SingleSessionBatchSampler(sids, batch_size=4, drop_last=False, seed=0)),
      len(SingleSessionBatchSampler(sids, batch_size=4, drop_last=True,  seed=0)))

## Section 5 — Common errors

### `ValueError: batch_size ... mutually exclusive with batch_sampler`

Passed both to the DataLoader. With a `batch_sampler`, drop `batch_size`/`shuffle`/`drop_last` — it owns them (§2.4).

### Batches DO mix sessions

The `session_ids` you handed the sampler don't align with the dataset's example order (wrong length, or built from a different ordering than `__getitem__` uses). Always pass `ds.session_ids` from the same dataset object.

### Same shuffle every epoch

`set_epoch(epoch)` isn't being called in the training loop. Without it, the sampler stays on epoch 0's ordering forever.

### One tiny session dominates early batches (or gets dropped entirely)

Expected with `drop_last=True`: a session smaller than `batch_size` contributes zero full batches and is skipped. If every session must appear, use `drop_last=False` or raise `batch_size`.

### Class imbalance persists despite session balancing

Different problem: this sampler balances *sessions*, not *classes*. Per-class weighting is a separate mechanism (`inverse_frequency_class_weights` in the loss — [04.7](../04_architecture/04.7_weighted_classification_loss.ipynb)).

## Section 6 — Further reading

- [Samplers in torch.utils.data](https://pytorch.org/docs/stable/data.html#data-loading-order-and-sampler) — the sampler/batch_sampler contract.
- [DistributedSampler](https://pytorch.org/docs/stable/data.html#torch.utils.data.distributed.DistributedSampler) — where `set_epoch` comes from and why multi-GPU training needs it.
- [`src/neural_data_decoding/data/samplers.py`](../../src/neural_data_decoding/data/samplers.py) + [`tests/unit/test_samplers.py`](../../tests/unit/test_samplers.py) — implementation and its property tests.

Next up: **[03.4 K-fold stratification deep dive](03.4_kfold_stratification_deep_dive.ipynb)** — the recursive splitter that keeps every fold representative.